In [ ]:
# imports
import pandas as pd
import numpy as np
import sys
sys.path.append("../../src")

from point_deduplicator import process_data

In [ ]:
# USER CONFIGURATION 
# change these paths

DATA_PATH = "YOUR DATA PATH HERE"
MAPPING_PATH = "YOUR MAPPING PATH HERE"

df = pd.read_csv(DATA_PATH)
mapping_df = pd.read_csv(MAPPING_PATH)

print("Data preview:")
display(df.head())

print("Mapping preview:")
display(mapping_df.head(10))

print("\nColumns:")
print(df.columns.tolist())

In [ ]:
# USER CONFIGURATION
# Add here your column names

ID_COL = "Unique identifier in your dataset"
LIPAS_ID_COL = "Identifier from the spatial match (produced in QGIS, e.g. id or id_2)"
CLASS_COL = "Category of your data point"
TYPE_COL_LIPAS = "Category of the matched LIPAS point (e.g. tyyppi_nimi_fi)"

MAP_KEY = "Column in the mapping table representing your dataset category"
MAP_VALUE = "Corresponding LIPAS category in the mapping table"

NAME1_COL = "Name of your data point"      
NAME2_COL = "Name of the matched LIPAS point (from spatial search, e.g. nimi_fi)"      

In [ ]:
true_matches, true_matches_duplicates, mismatches, review_mismatches_duplicates, needs_checking, not_matched = process_data(
    df,
    mapping_df,
    id_col=ID_COL,
    lipas_id_col=LIPAS_ID_COL,
    class_col=CLASS_COL,
    type_col=TYPE_COL_LIPAS,
    map_key=MAP_KEY,
    map_value=MAP_VALUE,
    name1_col=NAME1_COL,
    name2_col=NAME2_COL
)

In [ ]:
# Review that the results are correct

true_matches.loc[:, [ID_COL, CLASS_COL, TYPE_COL_LIPAS, NAME1_COL, NAME2_COL]]

In [ ]:
# Review duplicate entries

duplicated_ids = review_mismatches_duplicates[ID_COL][review_mismatches_duplicates[ID_COL].duplicated(keep=False)]

duplicates_df = review_mismatches_duplicates[review_mismatches_duplicates['gid'].isin(duplicated_ids)]

duplicates_df.loc[:, [ID_COL, CLASS_COL, TYPE_COL_LIPAS, NAME1_COL, NAME2_COL]]

In [ ]:
# Manipulate results 
# Here for example I am moving a row from true_matches to mismatches because it wasn't really a true match

mismatches = pd.concat([mismatches, true_matches[(true_matches[NAME1_COL] == "Name of your data point") & (true_matches[NAME2_COL] == "Name of the matched LIPAS point")]], ignore_index=True)

In [ ]:
# after manipulation remeber to remove the moved rows from the original dataframes

true_matches = true_matches[~true_matches[ID_COL].isin(mismatches[ID_COL].unique())]
true_matches.shape

In [ ]:
# When you are sure of your mismatching rows you can add them to the not_matched df
# to represent that a valid match was not found for the entry

not_matched = pd.concat([not_matched, mismatches], axis=0, ignore_index=True)
not_matched = pd.concat([not_matched, needs_checking], axis=0, ignore_index=True)
not_matched = pd.concat([not_matched, review_mismatches_duplicates], axis=0, ignore_index=True)

In [ ]:
# Then you can mark the columns as nan to represent that a match was not found

not_matched[[ID_COL, TYPE_COL_LIPAS, NAME2_COL, 'mapped_lipas_category', 'category_match']] = np.nan
not_matched.head()

In [ ]:
# After you have reviewed the results and are sure of your matches and mismatches you can make the final matching df

final_matching_df =  pd.concat([not_matched, true_matches], axis=0, ignore_index=True)
final_matching_df

In [ ]:
# Then you can take a subset of columns of the final matching so it is easy to use with a database table for example
# Here is your dataset identificator column and the matched Lipas entry's identificator column

final_matching_df = final_matching_df[[ID_COL, LIPAS_ID_COL]].copy()
final_matching_df

In [ ]:
final_matching_df[[ID_COL, LIPAS_ID_COL]] = final_matching_df[[ID_COL, LIPAS_ID_COL]].astype("Int64")
final_matching_df.dtypes

In [ ]:
# Those can be renamed for clarity if wished

final_matching_df.rename(columns={LIPAS_ID_COL: "mapped_lipas_id", ID_COL: "my_id"}, inplace=True)
final_matching_df

In [ ]:
# Let's fill the nans as 0 because in our data model 0 represents no match found

final_matching_df["mapped_lipas_id"] = final_matching_df["mapped_lipas_id"].fillna(0)
final_matching_df

In [ ]:
# Then you can save results to your device if wanted

final_matching_df.to_csv("final_matching_df.csv", index=False, na_rep="NULL")